In [1]:
%load_ext autoreload
%autoreload 2
%env CUDA_VISIBLE_DEVICES=0
import os, sys
import time
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import lib
import torch, torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

experiment_name = 'UniMiB_data'
experiment_name = '{}_{}.{:0>2d}.{:0>2d}_{:0>2d}-{:0>2d}'.format(experiment_name, *time.gmtime()[:5])
print("experiment:", experiment_name)

env: CUDA_VISIBLE_DEVICES=0
experiment: UniMiB_data_2026.03.24_09-27


In [3]:
data = lib.Dataset("UniMiB", random_state=1337, quantile_transform=True, quantile_noise=1e-3)
in_features = data.X_train.shape[1]

mu, std = data.y_train.mean(), data.y_train.std()
normalize = lambda x: ((x - mu) / std).astype(np.float32)
data.y_train, data.y_valid, data.y_test = map(normalize, [data.y_train, data.y_valid, data.y_test])

print("mean = %.5f, std = %.5f" % (mu, std), f"in_features = {in_features}")

mean = 0.07834, std = 0.72493 in_features = 6


In [ ]:
# model = nn.Sequential(
#     lib.DenseBlock(in_features, 32, num_layers=3, tree_dim=1, depth=3, flatten_output=False,choice_function=lib.entmax15, bin_function=lib.entmoid15),
#     lib.Lambda(lambda x: x[..., 0].mean(dim=-1)),  # average first channels of every tree
    
# ).to(device)

model = nn.Sequential(
    lib.DenseBlock(
        in_features,
        8,
        num_layers=1,
        tree_dim=1,
        depth=2,
        flatten_output=False
    ),
    lib.Lambda(lambda x: x.mean(dim=-1).mean(dim=-1)),
    #lib.Lambda(lambda x: x.mean(dim=-1)),
).to(device)

with torch.no_grad():
    res = model(torch.as_tensor(data.X_train[:128], device=device))
    # trigger data-aware init
    
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

c:\All doc\SDU\Semester 04\code\node-master\notebooks\..\lib\odst.py:113: UserWarning: Data-aware initialization is performed on less than 1000 data points. This may cause instability.To avoid potential problems, run this model on a data batch with at least 1000 data samples.You can do so manually before training. Use with torch.no_grad() for memory efficiency.
  warn("Data-aware initialization is performed on less than 1000 data points. This may cause instability."


In [5]:
from qhoptim.pyt import QHAdam
optimizer_params = { 'nus':(0.7, 1.0), 'betas':(0.95, 0.998) }

In [6]:
trainer = lib.Trainer(
    model=model, loss_function=F.mse_loss,
    experiment_name=experiment_name,
    warm_start=False,
    Optimizer=QHAdam,
    optimizer_params=optimizer_params,
    verbose=True,
    n_last_checkpoints=5
)

In [7]:
from tqdm import tqdm
from IPython.display import clear_output
loss_history, mse_history = [], []
best_mse = float('inf')
best_step_mse = 0
early_stopping_rounds = 5000
report_frequency = 100

In [8]:
#epochs=float('inf')
epochs = 10
batch_size = 1024
batch_size_mse = 16384
# batches = lib.iterate_minibatches(data.X_train, data.y_train, batch_size=batch_size, shuffle=True, epochs=epochs)
# print("Number of batches:", len(list(batches)))
report_frequency = 1
for batch in lib.iterate_minibatches(data.X_train, data.y_train, batch_size=batch_size, shuffle=True, epochs=epochs):
    print("Number of batches:", trainer.step)
    metrics = trainer.train_on_batch(*batch, device=device)
    
    # FIX: convert tensor to number
    loss_history.append(metrics['loss'].item())

    if trainer.step % report_frequency == 0:
        trainer.save_checkpoint()
        trainer.average_checkpoints(out_tag='avg')
        trainer.load_checkpoint(tag='avg')

        mse = trainer.evaluate_mse(data.X_valid, data.y_valid, device=device, batch_size=batch_size_mse)

        if mse < best_mse:
            best_mse = mse
            best_step_mse = trainer.step
            trainer.save_checkpoint(tag='best_mse')

        mse_history.append(mse)
        
        trainer.load_checkpoint()  # last
        trainer.remove_old_temp_checkpoints()

        clear_output(True)

        plt.figure(figsize=(18, 6))

        plt.subplot(1, 2, 1)
        plt.plot(loss_history)
        plt.title('Loss')
        plt.grid()

        plt.subplot(1, 2, 2)
        plt.plot(mse_history)
        plt.title('MSE')
        plt.grid()

        plt.show()

        print("Loss %.5f" % metrics['loss'].item())
        print("Val MSE: %0.5f" % mse)

    if trainer.step > best_step_mse + early_stopping_rounds:
        print('BREAK. There is no improvement for {} steps'.format(early_stopping_rounds))
        print("Best step:", best_step_mse)
        print("Best Val MSE: %0.5f" % best_mse)
        break


Number of batches: 0


c:\All doc\SDU\Semester 04\code\node-master\notebooks\..\lib\trainer.py:123: UserWarning: Using a target size (torch.Size([1024])) that is different to the input size (torch.Size([1024, 8])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = self.loss_function(self.model(x_batch), y_batch).mean()


RuntimeError: The size of tensor a (8) must match the size of tensor b (1024) at non-singleton dimension 1

In [ ]:
trainer.load_checkpoint(tag='best_mse')
mse = trainer.evaluate_mse(data.X_test, data.y_test, device=device)
print('Best step: ', trainer.step)
print("Test MSE: %0.5f" % (mse))

In [ ]:
mse * std ** 2